# Amazon Laptop Market Analysis
This notebook scrapes Amazon laptop data, cleans it, analyzes it, and saves results.

In [1]:
!pip install requests beautifulsoup4 pandas lxml


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import random


In [3]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9"
}

BASE_URL = "https://www.amazon.in/s?k=laptop&page={}"

data = []


In [4]:
def extract(card):

    page_text = card.get_text(" ", strip=True)

    # product_title
    title_tag = card.find("h2")
    title = title_tag.get_text(strip=True) if title_tag else None

    # brand
    brand = re.match(r"^\w+", title).group() if title else None

    # price
    price_match = re.search(r"₹\s?([\d,]+)", page_text)
    price = price_match.group(1).replace(",", "") if price_match else None

    # rating
    rating_match = re.search(r"(\d+(\.\d+)?)\s?out of 5", page_text)
    rating = rating_match.group(1) if rating_match else None

    # processor
    processor_match = re.search(r"(i[3579]|Ryzen\s?\d)", page_text)
    processor = processor_match.group() if processor_match else None

    # RAM
    ram_match = re.search(r"(\d+)\s?GB\s?(RAM)?", page_text)
    ram = ram_match.group(1)+"GB" if ram_match else None

    # storage
    storage_match = re.search(r"(\d+)\s?(GB|TB)\s?(SSD|HDD)?", page_text)
    storage = storage_match.group() if storage_match else None

    # screen size
    screen_match = re.search(r"(\d+(\.\d+)?)\s?(Inch|Inches)", page_text)
    screen = screen_match.group(1) if screen_match else None

    # colour
    color_match = re.search(r"(?:Colour|Color)\s*[:-]?\s*([A-Za-z ]+)", page_text)
    color = color_match.group(1).strip() if color_match else None

    # windows version
    win_match = re.search(r'(?:Windows\s*|Win\s*)(\d+)', page_text, re.I)
    windows = "Windows " + win_match.group(1) if win_match else None

    return [title, brand, ram, storage, windows, color, price, rating, processor, screen]


In [5]:
for page in range(1,8):

    print("Page:", page)

    url = BASE_URL.format(page)
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "lxml")

    cards = soup.select('div[data-component-type="s-search-result"]')

    for card in cards:
        row = extract(card)
        if row[0]:  # must have title
            data.append(row)

    time.sleep(random.uniform(2,4))

print("Scraping finished")


Page: 1
Page: 2
Page: 3
Page: 4
Page: 5
Page: 6
Page: 7
Scraping finished


In [6]:
columns = [
    "product_title","Brand","RAM","Storage SSD",
    "Windows","Color","Price","Rating","Processor","Screen"
]

df = pd.DataFrame(data, columns=columns)

df.to_csv("amazon_laptop_raw.csv", index=False)

print("Total rows:", len(df))
df.head()


Total rows: 154


,product_title,Brand,RAM,Storage SSD,Windows,Color,Price,Rating,Processor,Screen
0,"ASUS Vivobook S14,Smartchoice,AMD Ryzen AI 7 3...",ASUS,16GB,16GB,Windows 11,None,83990,3.8,None,None
1,"HP OmniBook 5 OLED (Previously Pavilion), Snap...",HP,16GB,16GB,Windows 11,None,66990,4.1,None,None
2,"BrowseBook 14.1"" FHD IPS Laptop | Best Student...",BrowseBook,4GB,4GB,Windows 11,None,12990,3.0,None,None
3,"HP 15, 13th Gen Intel Core i3-1315U (12GB DDR4...",HP,12GB,12GB,Windows 11,None,41990,4.1,i3,None
4,"EBook 11.6"" HD Laptop | Best Student & Office ...",EBook,4GB,4GB,Windows 11,None,10990,5.0,None,None


In [7]:
print(df.shape)
print(df.isnull().sum())

df = df.drop_duplicates()

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

df["Brand"] = df["Brand"].fillna(df["product_title"].str.split().str[0])

df.to_csv("amazon_laptop_cleaned.csv", index=False)

df.head()


(154, 10)
product_title      0
Brand              0
RAM               10
Storage SSD       10
Windows           20
Color            154
Price              1
Rating             4
Processor         76
Screen           145
dtype: int64


,product_title,Brand,RAM,Storage SSD,Windows,Color,Price,Rating,Processor,Screen
0,"ASUS Vivobook S14,Smartchoice,AMD Ryzen AI 7 3...",ASUS,16GB,16GB,Windows 11,None,83990.0,3.8,None,None
1,"HP OmniBook 5 OLED (Previously Pavilion), Snap...",HP,16GB,16GB,Windows 11,None,66990.0,4.1,None,None
2,"BrowseBook 14.1"" FHD IPS Laptop | Best Student...",BrowseBook,4GB,4GB,Windows 11,None,12990.0,3.0,None,None
3,"HP 15, 13th Gen Intel Core i3-1315U (12GB DDR4...",HP,12GB,12GB,Windows 11,None,41990.0,4.1,i3,None
4,"EBook 11.6"" HD Laptop | Best Student & Office ...",EBook,4GB,4GB,Windows 11,None,10990.0,5.0,None,None


In [8]:
print("Highest Avg Price Brand")
print(df.groupby("Brand")["Price"].mean().sort_values(ascending=False).head(1))

print("\nHighest Avg Rating Brand")
print(df.groupby("Brand")["Rating"].mean().sort_values(ascending=False).head(1))

print("\nTop 5 Most Reviewed (if available)")
if "Reviews" in df.columns:
    print(df.sort_values("Reviews",ascending=False).head(5))
else:
    print("Reviews column not scraped")

print("\nPrice vs Rating Correlation")
print(df["Price"].corr(df["Rating"]))

print("\nRAM Distribution")
print(df["RAM"].value_counts())

print("\nMost Common Screen Size")
print(df["Screen"].value_counts().head(1))

print("\nLaptops Under ₹50,000")
print(len(df[df["Price"] < 50000]))


Highest Avg Price Brand
Brand
MSI    207501.0
Name: Price, dtype: float64

Highest Avg Rating Brand
Brand
New    5.0
Name: Rating, dtype: float64

Top 5 Most Reviewed (if available)
Reviews column not scraped

Price vs Rating Correlation
-0.17761492637993234

RAM Distribution
RAM
16GB     54
8GB      29
6GB       9
4GB       6
32GB      5
12GB      3
24GB      3
64GB      1
128GB     1
Name: count, dtype: int64

Most Common Screen Size
Screen
14.1    4
Name: count, dtype: int64

Laptops Under ₹50,000
57


In [9]:
if len(df) < 50:
    print("Warning: Too few results. Amazon may be blocking requests.")
